# Quickstart

ETHOS.TSAM compresses a long time series into a few **typical periods**. Here we
take six weeks of hourly load and shrink it to **6 typical days**, then check that
the result still looks like the original.

We use a six-week slice so the plots stay readable — the very same call works on a
full multi-year dataset.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data
data.shape

## The data

Six weeks is hard to read as a line. As a **heatmap** it fits in one picture — one
column per day, hours running down — so the daily shape and the slow drift over the
weeks are both visible at a glance:

In [ ]:
def day_heatmap(frame, column, title):
    by_day = tsam.unstack_to_periods(frame[[column]], period_duration="1D")[column]
    return px.imshow(
        by_day.values.T,
        aspect="auto",
        origin="lower",
        labels={"x": "day", "y": "hour", "color": column},
        title=title,
    )


day_heatmap(data, "Load", "Load — every day in the six-week window")

## One call

Reduce the six weeks to 6 typical days:

In [ ]:
result = tsam.aggregate(data, n_clusters=6, period_duration="1D")

## Members → representatives

Each real day belongs to a group; the days in a group are its **members**. You can
think of every day as a candidate for its group — tsam then represents the whole
group with one **typical day**. Below, the faint lines are the member days and the
bold line is the typical day standing in for them — use the slider to step through
the groups:

In [ ]:
result.plot.cluster_members(columns=["Load"])

The six typical days together, each labelled with how many real days it represents:

In [ ]:
result.plot.cluster_representatives(columns=["Load"])

## Did it hold up?

Expand the typical days back over the whole window and compare. As heatmaps the
full window is visible at once — the reconstruction keeps the daily structure:

In [ ]:
day_heatmap(
    result.reconstructed, "Load", "Reconstructed Load — typical days expanded back out"
)

Zoom into a single week to see the detail (a line plot on a short slice stays readable):

In [ ]:
week = slice("2010-01-11", "2010-01-17")
result.plot.compare(
    columns=["Load"],
    time_slice=week,
    color="source",
    title="One week: original vs. reconstructed Load",
)

And the error as a single number per column:

In [ ]:
result.accuracy.rmse

## Where to go from here

That's the whole core workflow. From here you can:

- **Use fewer time steps** — raise or lower `n_clusters`, or add
  `segments=SegmentConfig(...)` to coarsen the resolution within each day. See
  [How small can you go?](tuning.ipynb).
- **Preserve peaks or shapes** — choose a representation or force extreme days.
  See [Representations](representations.ipynb).
- **Feed an optimization** — `result.cluster_representatives`,
  `result.cluster_counts`, and `result.disaggregate(...)` are what your model
  needs. See [Optimization workflow](optimization_workflow.ipynb).